# Table of Contents
- Install Dependencies and Utilities
- You Only Look Ones (YOLO)
- Object Detection
- Image Segmentation
- Pose Detection
- Image Classification
- Oriented Object Detection using OBB
- Object Tracking in Video
- Object Counting from Video

# Install Dependencies and Utilities

In [ ]:
!pip install -q torch torchvision supervision ultralytics roboflow opencv-python

In [ ]:
!pip install --upgrade -q "pillow<11.0.0"

In [ ]:
%pip install -q ffmpeg-python

import ffmpeg
from pathlib import Path

def convert_to_web_video(input_path: str, output_path: str, crf: int = 23, preset: str = 'fast') -> bool:
    """
    Converts a video to a browser-compatible H.264/AAC format using ffmpeg-python.

    Parameters:
        input_path (str): Path to the source video.
        output_path (str): Path where the web-optimized video will be saved.
        crf (int): Constant Rate Factor for quality (18-28 is typical; lower is better quality).
        preset (str): Encoding speed preset (e.g., 'fast', 'medium', 'slow').

    Returns:
        bool: True if successful, False otherwise.
    """
    # Ensure the input file exists before calling FFmpeg
    if not Path(input_path).exists():
        print(f"Error: Input file not found at {input_path}")
        return False

    try:
        print(f"Converting {input_path} to web-supported format...")

        (
            ffmpeg
            .input(input_path)
            .output(
                output_path,
                vcodec='libx264',
                preset=preset,
                crf=crf,
                pix_fmt='yuv420p',
                acodec='aac'
            )
            .overwrite_output()
            .run(capture_stdout=True, capture_stderr=True)
        )

        print(f"Successfully converted! Saved to: {output_path}")
        return True

    except ffmpeg.Error as e:
        print("FFmpeg conversion failed!")
        # Decode and print the actual FFmpeg CLI error log
        if e.stderr:
            print(e.stderr.decode('utf-8'))
        return False

In [ ]:
%pip install -q ultralytics
import ultralytics
ultralytics.checks()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import requests
from ultralytics import YOLO

# You Only Look Onece (YOLO)

**YOLO** is a **CNN-based** real-time object detection model that processes an entire image in a **single forward pass** through a convolutional neural network. It predicts bounding boxes, object classes, and confidence scores directly from the image, making it fast enough for real-time applications like surveillance, autonomous vehicles, and video analytics.

![](https://editor.analyticsvidhya.com/uploads/1512812.png)
![](https://media.geeksforgeeks.org/wp-content/uploads/20200305122535/final_result.jpg)

But **modern YOLO models (e.g. YOLOv11)**  are no longer only object detection models. It can perform tasks such as object detection, instance segmentation, image classification, pose estimation, and other vision tasks.

![](https://storage.ghost.io/c/2c/8d/2c8d8c0d-1c15-4b6d-825e-02b78d61d40a/content/images/2024/11/Screenshot-2024-11-13-at-11.14.35-AM.png)

# Object Detection

Object detection is a foundational computer vision technique that identifies and locates objects within images or videos. It goes beyond simple image classification by simultaneously predicting an object's category (e.g., "car", "pedestrian") and its exact spatial coordinates, usually visualized as a rectangular bounding box

![](https://upload.wikimedia.org/wikipedia/commons/3/38/Detected-with-YOLO--Schreibtisch-mit-Objekten.jpg)

![](https://iq.opengenus.org/content/images/2021/11/image-classification-vs-object-detection.png)

![](https://www.digitalocean.com/api/static-content/v1/images?src=https%3A%2F%2Fdoimages.nyc3.cdn.digitaloceanspaces.com%2F010AI-ML%2F2025%2FShaoni%2FAdrien%2F17-sept%2Fimage_1.png&token=6b21cc8876d3c3d14a314eea961fab57e6ae32ac1b7ec3335b27c7b3e7aa3d27)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import requests
from ultralytics import YOLO

# 1. Image source
img_url = 'https://ultralytics.com/images/bus.jpg'

# 2. Download and open the original image
original_img = Image.open(requests.get(img_url, stream=True).raw)

# 3. Load the pre-trained Object Detection model
model = YOLO('yolo11n.pt')

# 4. Predict (passing the already downloaded original_img)
results = model.predict(source=original_img)

# 5. Process the detection overlay results (bounding boxes)
annotated_img_bgr = results[0].plot()
annotated_img_rgb = annotated_img_bgr[:, :, ::-1]  # Convert BGR to RGB for Matplotlib

# 6. Display the images side-by-side
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Left subplot: Original Image
axes[0].imshow(original_img)
axes[0].set_title("Original Image", fontsize=14)
axes[0].axis('off')

# Right subplot: Object Detection Result
axes[1].imshow(annotated_img_rgb)
axes[1].set_title("Object Detection Result", fontsize=14)
axes[1].axis('off')

plt.tight_layout()
plt.show()

# Image Segmentation

Image segmentation is a computer vision technique that partitions a digital image into multiple segments or regions.

**Key Types of Segmentation**

- **Semantic Segmentation:** Assigns a class label (e.g., "car," "road," "tree") to every pixel. All objects of the same class are treated identically.

- **Instance Segmentation:** Detects and delineates each distinct object of a class separately. For example, if there are two people, it will label them as distinct entities ("person 1" and "person 2").


![](https://datahacker.rs/wp-content/uploads/2021/11/image-19-1024x313.png)


**Segmentaion Mask:** A segmentation mask is a pixel-level overlay used in computer vision that outlines the exact shape of an object within an image.

![](https://raw.githubusercontent.com/dmlc/web-data/master/mxnet/doc/tutorials/data_aug/outputs/with_mask/masks.png)


A segmentation model does not output a 1D probability array like a normal classifier. It outputs a mask (spatial prediction).

![](https://miro.medium.com/v2/resize:fit:2000/1*s4MMIUF0QECVP0UGQWzhxg.png)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import requests
from ultralytics import YOLO

# 1. Image source
img_url = 'https://ultralytics.com/images/bus.jpg'

# 2. Download and open the original image
original_img = Image.open(requests.get(img_url, stream=True).raw)

# 3. Load the pre-trained Segmentation model
model = YOLO('yolo11n-seg.pt')

# 4. Predict (passing the already downloaded original_img)
results = model.predict(source=original_img)

# 5. Process the segmentation overlay results
annotated_img_bgr = results[0].plot(boxes=False)
annotated_img_rgb = annotated_img_bgr[:, :, ::-1]  # Convert BGR to RGB for Matplotlib

# 6. Display the images side-by-side
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Left subplot: Original Image
axes[0].imshow(original_img)
axes[0].set_title("Original Image", fontsize=14)
axes[0].axis('off')

# Right subplot: Segmentation Result
axes[1].imshow(annotated_img_rgb)
axes[1].set_title("Instance Segmentation Result", fontsize=14)
axes[1].axis('off')

plt.tight_layout()
plt.show()

# Pose Estimation

Pose estimation is a computer vision technique that predicts the **position** and **orientation** of a person or object in an image or video. It identifies specific **anatomical landmarks (like joints)** to create a digital **skeleton map**, enabling machines to understand human postures, movements, and actions.

![](https://cdn.analyticsvidhya.com/wp-content/uploads/2022/01/65603fig-67bc4bba7b7dc.webp)



In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import requests
from ultralytics import YOLO

# 1. Image source
img_url = 'https://ultralytics.com/images/bus.jpg'

# 2. Download and open the original image
original_img = Image.open(requests.get(img_url, stream=True).raw)

# 3. Load the pre-trained Pose estimation model
model = YOLO('yolo11n-pose.pt')

# 4. Predict (passing the already downloaded original_img)
results = model.predict(source=original_img)

# 5. Process the skeleton overlay results
# We extract the first result and plot it without bounding boxes
annotated_img_bgr = results[0].plot(boxes=False)
annotated_img_rgb = annotated_img_bgr[:, :, ::-1]  # Convert BGR to RGB for Matplotlib

# 6. Display the images side-by-side
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Left subplot: Original Image
axes[0].imshow(original_img)
axes[0].set_title("Original Image", fontsize=14)
axes[0].axis('off')

# Right subplot: Pose Estimation Result
axes[1].imshow(annotated_img_rgb)
axes[1].set_title("Pose Estimation Result", fontsize=14)
axes[1].axis('off')

plt.tight_layout()
plt.show()

# Image Classification

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import requests
from ultralytics import YOLO

# 1. Image source
img_url = 'https://ultralytics.com/images/bus.jpg'

# 2. Download and open the original image
original_img = Image.open(requests.get(img_url, stream=True).raw)

# 3. Load the pre-trained Image Classification model
model = YOLO('yolo11n-cls.pt')

# 4. Predict (passing the already downloaded original_img)
results = model.predict(source=original_img)

# 5. Process the classification overlay results (displays top predicted labels)
annotated_img_bgr = results[0].plot()
annotated_img_rgb = annotated_img_bgr[:, :, ::-1]  # Convert BGR to RGB for Matplotlib

# 6. Display the images side-by-side
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Left subplot: Original Image
axes[0].imshow(original_img)
axes[0].set_title("Original Image", fontsize=14)
axes[0].axis('off')

# Right subplot: Classification Result
axes[1].imshow(annotated_img_rgb)
axes[1].set_title("Classification Result", fontsize=14)
axes[1].axis('off')

plt.tight_layout()
plt.show()

# Oriented Object Detection using OBB

**Oriented Bounding Boxes (OBB)** include an extra angle parameter, allowing the detector to rotate the boxes to tightly fit objects that are tilted (which is incredibly useful for things like **aerial drone imagery**, **retail text**, or **angled objects**)

![](https://cdn.prod.website-files.com/680a070c3b99253410dd3df5/684d84f5f3a3145136e7f901_684706d144fc80bc6b6a74fa_OBB-guide_Fig%25202.webp)

![](https://www.researchgate.net/publication/350522998/figure/fig6/AS:11431281211996974@1702531576752/Horizontal-bounding-boxes-HBB-and-oriented-bounding-boxes-OBB-a-HBB-is-widely.tif)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import requests
from ultralytics import YOLO

# 1. Image source
img_url = 'https://ultralytics.com/images/boats.jpg'

# 2. Download and open the original image
original_img = Image.open(requests.get(img_url, stream=True).raw)

# 3. Load the pre-trained Oriented Object Detection (OBB) model
model = YOLO('yolo11n-obb.pt')

# 4. Predict (passing the already downloaded original_img)
results = model.predict(source=original_img)

# 5. Process the rotated bounding boxes overlay results
annotated_img_bgr = results[0].plot()
annotated_img_rgb = annotated_img_bgr[:, :, ::-1]  # Convert BGR to RGB for Matplotlib

# 6. Display the images side-by-side
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Left subplot: Original Image
axes[0].imshow(original_img)
axes[0].set_title("Original Image", fontsize=14)
axes[0].axis('off')

# Right subplot: Oriented Object Detection Result
axes[1].imshow(annotated_img_rgb)
axes[1].set_title("Oriented Object Detection (OBB) Result", fontsize=14)
axes[1].axis('off')

plt.tight_layout()
plt.show()

# Object Tracking in Video

![](https://kajabi-storefronts-production.kajabi-cdn.com/kajabi-storefronts-production/file-uploads/blogs/22606/images/a05751f-a0ff-cb50-babe-ca164622c6b_ezgif.com-gif-maker_1_.gif)


**Object tracking** in computer vision is the process of **identifying and following the same object across multiple frames in a video**.

A tracking system answers:

* **Where is the object?** → bounding box/location
* **Which object is it?** → maintains a unique ID
* **Where is it moving?** → trajectory/motion

Example:

Video frames:

```
Frame 1:
        🚗
       ID:1

Frame 2:
          🚗
         ID:1

Frame 3:
            🚗
           ID:1
```

The car is detected in each frame, and the tracker understands it is **the same car**.

Typical pipeline:

```
Video Frame
     |
     v
Object Detection (YOLO)
     |
     v
Tracker (ByteTrack, DeepSORT, SORT)
     |
     v
Object IDs + Movement paths
```

Common tracking algorithms:

* **SORT** → Simple Online Realtime Tracking (uses Kalman Filter + IoU matching)
* **DeepSORT** → adds appearance features using a neural network
* **ByteTrack** → modern fast tracker that uses high/low confidence detections

Applications:

* 🚗 Vehicle counting
* 👥 People tracking
* ⚽ Sports analysis
* 🎥 Surveillance
* 🏭 Industrial monitoring

Difference:

**Detection:**

> "There is a car here in this frame."

**Tracking:**

> "This is the same car I saw 5 seconds ago, and it moved from here to there."


In [ ]:
# Install
# !pip install -q ultralytics supervision opencv-python


import cv2
import requests
import supervision as sv
from ultralytics import YOLO


# -----------------------------
# 1. Download video
# -----------------------------

video_url = "https://media.roboflow.com/supervision/video-examples/vehicles.mp4"

input_video = "/content/vehicles.mp4"

response = requests.get(video_url)

with open(input_video, "wb") as f:
    f.write(response.content)


# -----------------------------
# 2. Load YOLO
# -----------------------------

model = YOLO("yolo11n.pt")


# -----------------------------
# 3. Video information
# -----------------------------

video_info = sv.VideoInfo.from_video_path(
    input_video
)

print(video_info)


# -----------------------------
# 4. Tracker
# -----------------------------

tracker = sv.ByteTrack()


# -----------------------------
# 5. Annotators
# -----------------------------

box_annotator = sv.BoxAnnotator()

label_annotator = sv.LabelAnnotator()


# -----------------------------
# 6. Counting storage
# -----------------------------

vehicle_ids = set()


# COCO vehicle classes
VEHICLES = {
    2: "car",
    3: "motorcycle",
    5: "bus",
    7: "truck"
}


# -----------------------------
# 7. Frame processing
# -----------------------------

def process_frame(frame, frame_index):

    # Detect
    results = model(frame, verbose=False)[0]


    detections = sv.Detections.from_ultralytics(
        results
    )


    # Keep vehicles only

    mask = [
        class_id in VEHICLES
        for class_id in detections.class_id
    ]

    detections = detections[mask]


    # Track

    detections = tracker.update_with_detections(
        detections
    )


    # Count unique IDs

    for tracker_id in detections.tracker_id:
        vehicle_ids.add(tracker_id)


    # Labels

    labels = []

    for class_id, tracker_id in zip(
        detections.class_id,
        detections.tracker_id
    ):

        labels.append(
            f"{VEHICLES[class_id]} ID:{tracker_id}"
        )


    # Draw boxes

    annotated = box_annotator.annotate(
        scene=frame,
        detections=detections
    )


    annotated = label_annotator.annotate(
        scene=annotated,
        detections=detections,
        labels=labels
    )


    # Count text

    cv2.putText(
        annotated,
        f"Vehicle Count: {len(vehicle_ids)}",
        (30,50),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0,255,0),
        2
    )


    return annotated



# -----------------------------
# 8. Process video
# -----------------------------

output_video = "/content/tracked_vehicles.mp4"


sv.process_video(
    source_path=input_video,
    target_path=output_video,
    callback=process_frame,
    show_progress=True
)


print("DONE")
print("Total vehicles:", len(vehicle_ids))
print(output_video)

In [ ]:
# converting video to browser supported codec
# this will take some times

convert_to_web_video(
    input_path='/content/tracked_vehicles.mp4',
    output_path='/content/tracked_vehicles_web.mp4'
)

In [ ]:
from IPython.display import Video, display

display(
    Video(
        "/content/tracked_vehicles_web.mp4",
        embed=True,
        width=800,

    )
)

# Object Counting from Video

**Tracking-based counting using a line** is a common computer vision technique where we:

1. Detect objects in every video frame
2. Assign a unique ID to each object using a tracker
3. Check if an object's path crosses a predefined line
4. Increase the count only once per object ID

Pipeline:

```
Video
  |
  v
YOLO Detector (CNN-based)
  |
  v
Detected objects
  |
  v
Tracker (ByteTrack / DeepSORT)
  |
  v
Object IDs + trajectories
  |
  v
Line crossing logic
  |
  v
Count
```

Example:

```
              Counting Line
        -------------------------

Frame 1:                     |
                             |
                             |                       🚗
                             |                       ID=5

Frame 2:
                             |
                             |
                             |          🚗
                             |          ID=5

Frame 3:
                             |
                             |
                       🚗    |          
                       ID=5  |
                        
                        
```

When ID=5 crosses the line:

```
Total Cars = Total Cars + 1
```

---

### Simple logic

```python
counted_ids = set()

if object_crosses_line:

    if object_id not in counted_ids:
        count += 1
        counted_ids.add(object_id)
```

---

Common applications:

* 🚗 Vehicle counting at roads
* 👥 People entering/exiting buildings
* 🏭 Production line counting
* 🏟 Sports player movement analysis

Modern implementations often use:

**YOLO + ByteTrack + Line Zone**

because YOLO finds objects, and ByteTrack keeps the same ID across frames.


In [ ]:
import cv2
import matplotlib.pyplot as plt

video_path = "vehicles.mp4"

cap = cv2.VideoCapture(video_path)

ret, frame = cap.read()

cap.release()

# convert BGR -> RGB
frame_rgb = cv2.cvtColor(
    frame,
    cv2.COLOR_BGR2RGB
)

plt.figure(figsize=(12,8))
plt.imshow(frame_rgb)
plt.axis("on")
plt.show()

In [ ]:
import cv2
from IPython.display import display, Image


video_path = "vehicles.mp4"


# your test points
LINE_START = (300, 1250)
LINE_END   = (3600, 1250)


# read first frame
cap = cv2.VideoCapture(video_path)

ret, frame = cap.read()

cap.release()


# draw line
cv2.line(
    frame,
    LINE_START,
    LINE_END,
    (0, 255, 0),
    5
)


# draw points
cv2.circle(
    frame,
    LINE_START,
    8,
    (255,0,0),
    -1
)

cv2.circle(
    frame,
    LINE_END,
    8,
    (255,0,0),
    -1
)


# save
cv2.imwrite(
    "line_preview.jpg",
    frame
)


display(
    Image(
        filename="line_preview.jpg"
    )
)

In [ ]:
import cv2
import requests
import supervision as sv
from ultralytics import YOLO


# --------------------------
# Download video
# --------------------------

url = "https://media.roboflow.com/supervision/video-examples/vehicles.mp4"

video_path = "/content/vehicles.mp4"

open(video_path, "wb").write(
    requests.get(url).content
)


# --------------------------
# Model
# --------------------------

model = YOLO("yolov8n.pt")


video_info = sv.VideoInfo.from_video_path(
    video_path
)


# --------------------------
# Tracker
# --------------------------

tracker = sv.ByteTrack()


# --------------------------
# Line counter
# --------------------------

# adjust according to video
LINE_START = sv.Point(300, 1250)
LINE_END   = sv.Point(3600, 1250)


line_zone = sv.LineZone(
    start=LINE_START,
    end=LINE_END
)


line_annotator = sv.LineZoneAnnotator()

box_annotator = sv.BoxAnnotator()

label_annotator = sv.LabelAnnotator()



# vehicle classes COCO
VEHICLES = {
    2:"car",
    3:"motorcycle",
    5:"bus",
    7:"truck"
}



# --------------------------
# Frame processing
# --------------------------

def process_frame(frame, frame_index):


    result = model(
        frame,
        verbose=False
    )[0]


    detections = sv.Detections.from_ultralytics(
        result
    )


    # keep vehicles only
    mask = [
        cid in VEHICLES
        for cid in detections.class_id
    ]

    detections = detections[mask]


    # tracking

    detections = tracker.update_with_detections(
        detections
    )


    # trigger line counter

    line_zone.trigger(
        detections
    )


    labels = [
        f"{VEHICLES[cid]} #{tid}"
        for cid,tid in zip(
            detections.class_id,
            detections.tracker_id
        )
    ]


    frame = box_annotator.annotate(
        frame,
        detections
    )


    frame = label_annotator.annotate(
        frame,
        detections,
        labels
    )


    frame = line_annotator.annotate(
        frame,
        line_zone
    )


    # counts

    cv2.putText(
        frame,
        f"IN: {line_zone.in_count}",
        (30,50),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0,255,0),
        2
    )


    cv2.putText(
        frame,
        f"OUT: {line_zone.out_count}",
        (30,100),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0,255,0),
        2
    )


    return frame



# --------------------------
# Process
# --------------------------

output = "/content/vehicle_count_line.mp4"


sv.process_video(
    source_path=video_path,
    target_path=output,
    callback=process_frame,
    show_progress=True
)


print("Done")
print(
    "IN:",
    line_zone.in_count,
    "OUT:",
    line_zone.out_count
)

In [ ]:
# converting video to browser supported codec
# this will take some times

convert_to_web_video(
    input_path='/content/vehicle_count_line.mp4',
    output_path='/content/vehicle_count_line_web.mp4'
)

In [ ]:
from IPython.display import Video, display

display(
    Video(
        "/content/vehicle_count_line_web.mp4",
        embed=True,
        width=800,

    )
)